# Band Padding Ablation Analysis

Compare zero-filled channel padding (full S2 input) against the standard baseline (BandFilter only).

- **Padding**: encoder receives all S2 bands; missing ones are zero-filled after normalization
- **Baseline**: standard UPerNet runs from `pangaea_full_seg/upernet/` (BandFilter, no padding)

In [ ]:
import re
import os
from pathlib import Path
import pandas as pd
import numpy as np

EXPERIMENTS_BASE = Path("/dss/dsstbyfs02/pn49cu/pn49cu-dss-0001/niklas/experiments")

# _s2full variants (band padding experiment) map to same display name as base encoder
ENCODER_LABELS = {
    "dofa_s2full":               "DOFA",
    "dofav2_s2full":             "DOFAv2",
    "panopticon_s2full":         "Panopticon",
    "smarties_vitb_s2full":      "SMARTIES",
    "copernicus_fm_base_s2full": "Copernicus-FM",
    "dofa":               "DOFA",
    "dofav2":             "DOFAv2",
    "carl":               "CARL",
    "panopticon":         "Panopticon",
    "copernicus_fm_base": "Copernicus-FM",
    "smarties_vitb":      "SMARTIES",
    "dinov2_vitb14":      "DINOv2",
    "unet_encoder":       "UNet",
    "unet_encoder_mi":    "UNet",
    "vit_scratch":        "ViT-Scratch",
    "vit_scratch_mi":     "ViT-Scratch",
}

DATASET_LABELS = {
    "hlsburnscars":      "HLS-BurnScars",
    "sen1floods11":      "Sen1Floods11",
    "mados":             "MADOS",
    "pastis":            "PASTIS",
    "dynamicen":         "DynamicEarthNet",
    "croptypemapping":   "CropTypeMapping",
    "spacenet7cd":       "SpaceNet7-CD",
    "ai4smallfarms":     "AI4SmallFarms",
    "fivebillionpixels": "FiveBillionPixels",
}

DATASET_ORDER = list(DATASET_LABELS.keys())

# _s2full variants must appear before their base counterparts for greedy matching
KNOWN_ENCODERS = [
    "copernicus_fm_base_s2full", "copernicus_fm_base", "copernicus_fm_large",
    "smarties_vitb_s2full", "smarties_vitb", "dinov2_vitb14",
    "panopticon_s2full",
    "dofav2_s2full", "dofa_s2full",
    "dofav2", "dofa", "panopticon", "carl",
    "vit_scratch_mi", "vit_scratch",
    "unet_encoder_mi", "unet_encoder",
]

KNOWN_DECODERS = [
    "seg_upernet_mt_ltae", "seg_upernet",
    "seg_siamupernet_conc", "seg_siamupernet_diff",
    "seg_siam_mlp_fusion", "seg_mlp_fusion",
    "seg_siamunet_conc", "seg_siamunet_diff", "seg_unet",
    "seg_linear_probe", "seg_siam_linear_probe",
    "cls_linear_probe",
]

DATASET_RE = re.compile(
    r"_(" + "|".join(re.escape(d) for d in DATASET_ORDER) + r")_"
)

# Ablation encoders in display order
ABL_ENCODERS_DISPLAY = ["DOFA", "DOFAv2", "CARL", "Panopticon", "SMARTIES", "Copernicus-FM"]

print("Config loaded.")

In [ ]:
def parse_run_dir_name(name: str) -> dict:
    """Extract dataset, encoder, decoder from a run directory name."""
    m = DATASET_RE.search(name)
    if not m:
        return {}
    dataset = m.group(1)
    rest = name[m.end():]

    encoder = next((e for e in KNOWN_ENCODERS if rest.startswith(e + "_")), None)
    if encoder:
        rest = rest[len(encoder) + 1:]

    decoder = next((d for d in KNOWN_DECODERS if rest.startswith(d)), None)

    return {"dataset": dataset, "encoder": encoder or "", "decoder": decoder or ""}


def parse_log_metrics(log_path: Path) -> dict:
    """Parse train.log-0 for test metrics (last test block)."""
    if not log_path.exists():
        return {}

    text = log_path.read_text(errors="ignore")
    sections = re.split(r"\[test\] ------- IoU", text)
    if len(sections) < 2:
        return {}
    test_block = "[test] ------- IoU" + sections[-1]

    metrics = {}
    metric_map = {
        "IoU":       "test_mIoU",
        "F1-score":  "test_mF1",
        "Precision": "test_mPrecision",
        "Recall":    "test_mRecall",
    }
    for section_name, metric_key in metric_map.items():
        pattern = rf"\[test\] ------- {section_name}.*?\[test\] Mean\s+([\d.]+)"
        m = re.search(pattern, test_block, re.DOTALL)
        if m:
            metrics[metric_key] = float(m.group(1))
    return metrics


def load_experiment_folder(
    folder,
    encoder_map=None,
    dataset_map=None,
    metric="test_mIoU",
    keep_all_metrics=False,
):
    """
    Load all runs from an experiment folder and return a long-form DataFrame.
    Walks up to 2 levels deep to find run dirs containing train.log-0.
    """
    enc_map = {**ENCODER_LABELS, **(encoder_map or {})}
    ds_map  = {**DATASET_LABELS,  **(dataset_map  or {})}
    folder  = Path(folder)

    run_dirs = []
    for child in sorted(folder.iterdir()):
        if not child.is_dir():
            continue
        if (child / "train.log-0").exists():
            run_dirs.append(child)
        else:
            for grandchild in sorted(child.iterdir()):
                if grandchild.is_dir() and (grandchild / "train.log-0").exists():
                    run_dirs.append(grandchild)

    rows = []
    for run_dir in run_dirs:
        parsed = parse_run_dir_name(run_dir.name)
        if not parsed.get("dataset"):
            continue
        log_metrics = parse_log_metrics(run_dir / "train.log-0")
        if not log_metrics:
            continue

        enc_raw = parsed["encoder"]
        ds_raw  = parsed["dataset"]

        row = {
            "run_dir":     run_dir.name,
            "encoder_raw": enc_raw,
            "decoder":     parsed["decoder"],
            "dataset_raw": ds_raw,
            "encoder":     enc_map.get(enc_raw, enc_raw),
            "dataset":     ds_map.get(ds_raw, ds_raw),
        }
        if keep_all_metrics:
            row.update(log_metrics)
        elif metric in log_metrics:
            row[metric] = log_metrics[metric]
        else:
            continue
        rows.append(row)

    return pd.DataFrame(rows)


def to_pivot(
    df,
    metric="test_mIoU",
    index="encoder",
    columns="dataset",
    aggfunc="max",
    dataset_order=None,
    encoder_order=None,
    add_mean=True,
):
    """Pivot a long-form results DataFrame to encoder x dataset."""
    if metric not in df.columns:
        raise ValueError(f"Metric '{metric}' not in DataFrame: {list(df.columns)}")

    pivot = df.pivot_table(index=index, columns=columns, values=metric, aggfunc=aggfunc)

    if dataset_order is None:
        ordered_ds = [DATASET_LABELS[d] for d in DATASET_ORDER if DATASET_LABELS[d] in pivot.columns]
        remaining  = [c for c in pivot.columns if c not in ordered_ds]
        dataset_order = ordered_ds + remaining
    pivot = pivot.reindex(columns=[c for c in dataset_order if c in pivot.columns])

    if encoder_order is not None:
        pivot = pivot.reindex([e for e in encoder_order if e in pivot.index])

    if add_mean:
        pivot["Mean"] = pivot.mean(axis=1, skipna=True).round(3)

    return pivot.round(2)


print("Functions defined.")

## Band Padding Results

Zero-fills missing S2 channels so every encoder receives the full S2 band set.

In [ ]:
df_pad = load_experiment_folder(
    EXPERIMENTS_BASE / "ablations" / "band_padding",
    metric="test_mIoU",
)
print(f"{len(df_pad)} runs loaded from band_padding/")
df_pad.groupby(["encoder", "dataset"])["test_mIoU"].max().unstack()

In [ ]:
pivot_pad = to_pivot(df_pad, encoder_order=ABL_ENCODERS_DISPLAY)
pivot_pad

## Baseline (Standard UPerNet, BandFilter only)

Loaded from `pangaea_full_seg/upernet/`, filtered to the 6 ablation encoders.

In [ ]:
df_baseline = load_experiment_folder(
    EXPERIMENTS_BASE / "pangaea_full_seg" / "upernet",
    metric="test_mIoU",
)
# Keep only encoders also present in the padding experiment
df_baseline = df_baseline[df_baseline["encoder"].isin(ABL_ENCODERS_DISPLAY)]
print(f"{len(df_baseline)} runs loaded from upernet/ (ablation encoders only)")
pivot_baseline = to_pivot(df_baseline, encoder_order=ABL_ENCODERS_DISPLAY)
pivot_baseline

## Δ Padding − Baseline

Positive values mean padding improved performance; negative means it hurt.

In [ ]:
# Align columns (datasets present in both)
shared_cols = [c for c in pivot_pad.columns if c in pivot_baseline.columns and c != "Mean"]
delta = (pivot_pad[shared_cols] - pivot_baseline[shared_cols]).round(3)

# Reindex to consistent encoder order
delta = delta.reindex([e for e in ABL_ENCODERS_DISPLAY if e in delta.index])
delta["Mean Δ"] = delta.mean(axis=1, skipna=True).round(3)

# Highlight: green = positive, red = negative
def color_delta(val):
    if pd.isna(val):
        return ""
    color = "#c6efce" if val > 0 else ("#ffc7ce" if val < 0 else "")
    return f"background-color: {color}"

delta.style.applymap(color_delta)

In [ ]:
# Summary: how many datasets improved vs degraded per encoder
summary = pd.DataFrame({
    "Improved (Δ > 0)": (delta[shared_cols] > 0).sum(axis=1),
    "Degraded (Δ < 0)": (delta[shared_cols] < 0).sum(axis=1),
    "Neutral (Δ = 0)": (delta[shared_cols] == 0).sum(axis=1),
    "Mean Δ":           delta["Mean Δ"],
})
summary

In [ ]:
# Check which runs are still missing in the padding experiment
print("=== Missing padding runs ===")
missing = pivot_pad.drop(columns="Mean").isna().stack()
missing_rows = missing[missing].reset_index()[["encoder", "dataset"]]
if missing_rows.empty:
    print("None — all runs completed.")
else:
    print(missing_rows.to_string(index=False))

## Paper-style table: w/o padding (baseline) vs w/ padding

In [ ]:
def color_pct_cell(val):
    if not isinstance(val, str) or val == "—":
        return ""
    v = int(val.replace("%", "").replace("+", ""))
    if v > 0:
        return "color: #276221; font-weight: bold"
    if v < 0:
        return "color: #9c1e1e; font-weight: bold"
    return ""


# Datasets present in both pivots (excluding Mean column)
datasets = [c for c in pivot_baseline.columns if c in pivot_pad.columns and c != "Mean"]
base = pivot_baseline[datasets]
pad  = pivot_pad[datasets]

cols, data = [], {}
for ds in datasets:
    pct = ((pad[ds] - base[ds]) / base[ds] * 100).round(0)
    data[(ds, "w/o padding")] = base[ds].round(2)
    data[(ds, "w/ padding")]  = pad[ds].round(2)
    data[(ds, "% diff")]      = pct.apply(lambda v: f"{int(v):+d}%" if not pd.isna(v) else "—")
    cols += [(ds, "w/o padding"), (ds, "w/ padding"), (ds, "% diff")]

tbl = pd.DataFrame(data, columns=pd.MultiIndex.from_tuples(cols))
tbl = tbl.reindex([e for e in ABL_ENCODERS_DISPLAY if e in tbl.index])

pct_cols = [c for c in tbl.columns if c[1] == "% diff"]
display(
    tbl.style
       .map(color_pct_cell, subset=pct_cols)
       .set_table_styles([{"selector": "th", "props": [("text-align", "center")]}])
)